# Sprint 4 — Streamlit Forecasting App Deployment

## Overview
This sprint turns our champion XGBoost model (MAE=132.67) into a production-ready
Streamlit web application for Corporación Favorita store managers.

**What we'll build:**
- Interactive date picker for forecast cut-off
- Single-day and Next N days (1–30) autoregressive forecast
- Historical + forecast line plot (~6 months of actuals)
- CSV download of forecast results
- Clean modular project structure (app/, model/, data/)
- GitHub repository with README.md

**Champion model:** XGBoost (HyperOpt tuned) — Sprint 3
**Run ID:** e98efe56e2984d0aa6f0907458eca43f


## 1. Project Setup — Create Folder Structure


In [1]:
# === 1. CREATE PROJECT FOLDER STRUCTURE ===
import os
import pathlib

# Define project root — create in same directory as notebooks
PROJECT_ROOT = os.path.join(os.path.dirname(os.getcwd()), "corporacion_favorita")

folders = [
    "app",
    "model",
    "data",
    "models",
    "mlflow_results",
    "docs",
]

for folder in folders:
    path = os.path.join(PROJECT_ROOT, folder)
    pathlib.Path(path).mkdir(parents=True, exist_ok=True)
    # Create __init__.py for Python packages
    if folder in ["app", "model", "data"]:
        init_file = os.path.join(path, "__init__.py")
        if not os.path.exists(init_file):
            with open(init_file, "w") as f:
                f.write(f"# {folder}/__init__.py\n# Marks this folder as a Python package.\n")

print("Project structure created:")
for root, dirs, files in os.walk(PROJECT_ROOT):
    level = root.replace(PROJECT_ROOT, '').count(os.sep)
    indent = '  ' * level
    print(f"{indent}{os.path.basename(root)}/")
    subindent = '  ' * (level + 1)
    for file in files:
        print(f"{subindent}{file}")


Project structure created:
corporacion_favorita/
  app/
    __init__.py
  data/
    __init__.py
  docs/
  mlflow_results/
  model/
    __init__.py
  models/


## 2. Copy MLflow Artifacts & Champion Model to Project


In [2]:
# === 2. COPY MLFLOW ARTIFACTS & CHAMPION MODEL ===
import shutil
import os

# Source: current sprint3 folder (where mlruns/ was created)
# The mlruns folder is in the same directory where sprint3 notebook ran
SPRINT3_DIR = os.getcwd()  # current notebook directory
PROJECT_ROOT = os.path.join(os.path.dirname(os.getcwd()), "corporacion_favorita")

# ── Copy MLflow runs ───────────────────────────────────────────
src_mlruns = os.path.join(SPRINT3_DIR, "mlruns")
dst_mlruns = os.path.join(PROJECT_ROOT, "mlflow_results")

if os.path.exists(src_mlruns):
    if os.path.exists(dst_mlruns):
        shutil.rmtree(dst_mlruns)
    shutil.copytree(src_mlruns, dst_mlruns)
    print(f"✓ MLflow runs copied to: {dst_mlruns}")
else:
    print(f"✗ mlruns not found at: {src_mlruns}")
    print("  Make sure you run this from the same folder as sprint3 notebook!")

# ── Copy champion model artifacts ─────────────────────────────
artifacts = {
    "champion_xgboost.json": "models/champion_xgboost.json",
    "champion_features.json": "models/champion_features.json",
    "champion_scaler.pkl":   "models/scaler.pkl",
}

for src_name, dst_rel in artifacts.items():
    src = os.path.join(SPRINT3_DIR, src_name)
    dst = os.path.join(PROJECT_ROOT, dst_rel)
    if os.path.exists(src):
        shutil.copy2(src, dst)
        print(f"✓ Copied: {src_name} → {dst_rel}")
    else:
        print(f"✗ Not found: {src_name} (check sprint3 folder)")

# ── Copy time series CSV ───────────────────────────────────────
ts_csv = os.path.join(SPRINT3_DIR, "ts_store44_item1047679.csv")
if os.path.exists(ts_csv):
    shutil.copy2(ts_csv, os.path.join(PROJECT_ROOT, "models", "ts_store44_item1047679.csv"))
    print(f"✓ Copied: ts_store44_item1047679.csv → models/")
else:
    print(f"✗ Not found: ts_store44_item1047679.csv")

print(f"\nProject models/ folder contents:")
for f in os.listdir(os.path.join(PROJECT_ROOT, "models")):
    size = os.path.getsize(os.path.join(PROJECT_ROOT, "models", f))
    print(f"  {f}  ({size/1024:.1f} KB)")


✓ MLflow runs copied to: C:\Users\armin\PycharmProjects\corporacion_favorita\mlflow_results
✓ Copied: champion_xgboost.json → models/champion_xgboost.json
✓ Copied: champion_features.json → models/champion_features.json
✓ Copied: champion_scaler.pkl → models/scaler.pkl
✓ Copied: ts_store44_item1047679.csv → models/

Project models/ folder contents:
  champion_features.json  (0.2 KB)
  champion_xgboost.json  (108.6 KB)
  scaler.pkl  (0.5 KB)
  ts_store44_item1047679.csv  (29.7 KB)


## 3. Configuration — app/config.py


In [5]:
# === 3. CREATE app/config.py ===
import os

PROJECT_ROOT = os.path.join(os.path.dirname(os.getcwd()), "corporacion_favorita")
config_path  = os.path.join(PROJECT_ROOT, "app", "config.py")

config_code = """import os

# Project root
BASE_DIR = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))

# MLflow tracking store (local copy from Sprint 3)
# Must be an absolute file:/// URI
MLFLOW_TRACKING_URI = "file:///" + os.path.join(BASE_DIR, "mlflow_results").replace("\\\\", "/")

# Champion model URI (XGBoost - best model from Sprint 3)
CHAMPION_RUN_ID = "e98efe56e2984d0aa6f0907458eca43f"
MODEL_URI = f"runs:/{CHAMPION_RUN_ID}/xgb_best_model"

# Local fallback paths (used if MLflow load fails)
MODEL_PATH    = os.path.join(BASE_DIR, "models", "champion_xgboost.json")
SCALER_PATH   = os.path.join(BASE_DIR, "models", "scaler.pkl")
FEATURES_JSON = os.path.join(BASE_DIR, "models", "champion_features.json")
TS_CSV_PATH   = os.path.join(BASE_DIR, "models", "ts_store44_item1047679.csv")

# Model and series constants
STORE_ID  = 44
ITEM_ID   = 1047679
LAG_DAYS  = 30

# Feature columns (same order as training)
FEATURE_COLS = [
    "day_of_week", "month", "day_of_month", "week_of_year",
    "is_weekend", "quarter",
    "lag_1", "lag_7", "lag_14", "lag_30",
    "rolling_mean_7", "rolling_std_7", "rolling_mean_14",
]
"""

with open(config_path, "w", encoding="utf-8") as f:
    f.write(config_code)

print(f"Created: app/config.py")
print(f"  Path: {config_path}")

# Verify paths exist
import json, pickle
exec(open(config_path, encoding="utf-8").read())
print(f"\n  BASE_DIR:             {BASE_DIR}")
print(f"  MLFLOW_TRACKING_URI:  {MLFLOW_TRACKING_URI}")
print(f"  MODEL_URI:            {MODEL_URI}")
print(f"  MODEL_PATH exists:    {os.path.exists(MODEL_PATH)}")
print(f"  SCALER_PATH exists:   {os.path.exists(SCALER_PATH)}")
print(f"  FEATURES_JSON exists: {os.path.exists(FEATURES_JSON)}")
print(f"  TS_CSV_PATH exists:   {os.path.exists(TS_CSV_PATH)}")


Created: app/config.py
  Path: C:\Users\armin\PycharmProjects\corporacion_favorita\app\config.py


NameError: name '__file__' is not defined

## 4. Data Utilities — data/data_utils.py

In [6]:
# === 4. CREATE data/data_utils.py ===
import os

PROJECT_ROOT = os.path.join(os.path.dirname(os.getcwd()), "corporacion_favorita")
data_utils_path = os.path.join(PROJECT_ROOT, "data", "data_utils.py")

code = """import pandas as pd
import numpy as np
import sys, os
sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))
from app.config import TS_CSV_PATH, FEATURE_COLS, LAG_DAYS


def load_timeseries() -> pd.Series:
    df = pd.read_csv(TS_CSV_PATH, index_col="date", parse_dates=True)
    df = df.asfreq("D")
    return df["unit_sales"]


def create_features(series: pd.Series) -> pd.DataFrame:
    df = series.to_frame(name="unit_sales")
    df["day_of_week"]     = df.index.dayofweek
    df["month"]           = df.index.month
    df["day_of_month"]    = df.index.day
    df["week_of_year"]    = df.index.isocalendar().week.astype(int)
    df["is_weekend"]      = (df.index.dayofweek >= 5).astype(int)
    df["quarter"]         = df.index.quarter
    df["lag_1"]           = df["unit_sales"].shift(1)
    df["lag_7"]           = df["unit_sales"].shift(7)
    df["lag_14"]          = df["unit_sales"].shift(14)
    df["lag_30"]          = df["unit_sales"].shift(30)
    df["rolling_mean_7"]  = df["unit_sales"].shift(1).rolling(7).mean()
    df["rolling_std_7"]   = df["unit_sales"].shift(1).rolling(7).std()
    df["rolling_mean_14"] = df["unit_sales"].shift(1).rolling(14).mean()
    return df


def make_forecast_row(date: pd.Timestamp, history: pd.Series) -> pd.DataFrame:
    row = {
        "day_of_week":     date.dayofweek,
        "month":           date.month,
        "day_of_month":    date.day,
        "week_of_year":    date.isocalendar()[1],
        "is_weekend":      int(date.dayofweek >= 5),
        "quarter":         date.quarter,
        "lag_1":           history.iloc[-1]  if len(history) >= 1  else np.nan,
        "lag_7":           history.iloc[-7]  if len(history) >= 7  else np.nan,
        "lag_14":          history.iloc[-14] if len(history) >= 14 else np.nan,
        "lag_30":          history.iloc[-30] if len(history) >= 30 else np.nan,
        "rolling_mean_7":  history.iloc[-7:].mean()  if len(history) >= 7  else np.nan,
        "rolling_std_7":   history.iloc[-7:].std()   if len(history) >= 7  else np.nan,
        "rolling_mean_14": history.iloc[-14:].mean() if len(history) >= 14 else np.nan,
    }
    return pd.DataFrame([row])[FEATURE_COLS]
"""

with open(data_utils_path, "w", encoding="utf-8") as f:
    f.write(code)

print(f"Created: data/data_utils.py")
print(f"  Path: {data_utils_path}")
print(f"  Functions: load_timeseries(), create_features(), make_forecast_row()")


Created: data/data_utils.py
  Path: C:\Users\armin\PycharmProjects\corporacion_favorita\data\data_utils.py
  Functions: load_timeseries(), create_features(), make_forecast_row()


## 5. Model Utilities — model/model_utils.py


In [7]:
# === 5. CREATE model/model_utils.py ===
import os

PROJECT_ROOT = os.path.join(os.path.dirname(os.getcwd()), "corporacion_favorita")
model_utils_path = os.path.join(PROJECT_ROOT, "model", "model_utils.py")

code = """import os
import sys
import pandas as pd
import xgboost as xgb

# Fix for user-installed packages (mlflow, hyperopt)
sys.path.insert(0, os.path.expanduser('~\\\\AppData\\\\Roaming\\\\Python\\\\Python39\\\\site-packages'))
import mlflow

sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))
from app.config import MLFLOW_TRACKING_URI, MODEL_URI, MODEL_PATH

def load_champion_model():
    \"\"\"Loads the champion XGBoost model from MLflow, with local JSON fallback.\"\"\"
    # Try MLflow first
    try:
        mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
        model = mlflow.xgboost.load_model(MODEL_URI)
        print("Model loaded successfully from MLflow.")
        return model
    except Exception as e:
        print(f"MLflow load failed: {e}")
        print("Falling back to local model file...")

    # Fallback to local JSON
    try:
        model = xgb.XGBRegressor()
        model.load_model(MODEL_PATH)
        print("Model loaded successfully from local fallback.")
        return model
    except Exception as e:
        print(f"Failed to load local model: {e}")
        raise e

def predict_one(model, feature_row: pd.DataFrame) -> float:
    \"\"\"Makes a single prediction using the loaded model.\"\"\"
    if isinstance(model, xgb.Booster):
        dmatrix = xgb.DMatrix(feature_row)
        pred = model.predict(dmatrix)
        return float(pred[0])
    else:
        pred = model.predict(feature_row)
        return float(pred[0])
"""

with open(model_utils_path, "w", encoding="utf-8") as f:
    f.write(code)

print(f"Created: model/model_utils.py")
print(f"  Path: {model_utils_path}")
print(f"  Functions: load_champion_model(), predict_one()")


Created: model/model_utils.py
  Path: C:\Users\armin\PycharmProjects\corporacion_favorita\model\model_utils.py
  Functions: load_champion_model(), predict_one()


## 6. Streamlit Application — app/main.py


In [9]:
# === 6. CREATE app/main.py ===
import os

PROJECT_ROOT = os.path.join(os.path.dirname(os.getcwd()), "corporacion_favorita")
main_path = os.path.join(PROJECT_ROOT, "app", "main.py")

code = """import sys
import os

# Ensure project root is on path
PROJECT_ROOT = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
sys.path.insert(0, PROJECT_ROOT)

import streamlit as st
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from datetime import date, timedelta

from app.config import (
    STORE_ID, ITEM_ID, FEATURE_COLS, LAG_DAYS,
    MODEL_PATH, SCALER_PATH, FEATURES_JSON, TS_CSV_PATH
)
from data.data_utils import load_timeseries, make_forecast_row
from model.model_utils import load_champion_model

# ─────────────────────────────────────────────────────────────
# PAGE CONFIG
# ─────────────────────────────────────────────────────────────
st.set_page_config(
    page_title="Corporacion Favorita | Sales Forecast",
    page_icon="📦",
    layout="wide",
    initial_sidebar_state="expanded",
)

# ─────────────────────────────────────────────────────────────
# CUSTOM CSS — dark professional theme
# ─────────────────────────────────────────────────────────────
st.markdown(\"\"\"
<style>
    .main { background-color: #0e1117; }
    .block-container { padding-top: 1.5rem; }
    .metric-card {
        background: linear-gradient(135deg, #1e2130, #2a2f45);
        border-radius: 12px;
        padding: 1.2rem 1.5rem;
        border-left: 4px solid #4c9be8;
        margin-bottom: 0.5rem;
    }
    .metric-label { color: #8b9dc3; font-size: 0.78rem; text-transform: uppercase; letter-spacing: 0.08em; }
    .metric-value { color: #e8eaf6; font-size: 1.8rem; font-weight: 700; }
    .metric-sub   { color: #4c9be8; font-size: 0.75rem; margin-top: 0.2rem; }
    .section-header {
        color: #4c9be8;
        font-size: 1.05rem;
        font-weight: 600;
        text-transform: uppercase;
        letter-spacing: 0.06em;
        margin-bottom: 0.5rem;
        border-bottom: 1px solid #2a2f45;
        padding-bottom: 0.3rem;
    }
    .stDownloadButton > button {
        background: linear-gradient(90deg, #4c9be8, #3a7bd5);
        color: white;
        border: none;
        border-radius: 8px;
        padding: 0.5rem 1.5rem;
        font-weight: 600;
        width: 100%;
    }
    .stDownloadButton > button:hover { background: linear-gradient(90deg, #3a7bd5, #2c5faa); }
</style>
\"\"\", unsafe_allow_html=True)

# ─────────────────────────────────────────────────────────────
# CACHED LOADERS
# ─────────────────────────────────────────────────────────────
@st.cache_resource(show_spinner="Loading champion model...")
def get_model():
    return load_champion_model()

@st.cache_data(show_spinner="Loading historical data...")
def get_timeseries():
    return load_timeseries()

# ─────────────────────────────────────────────────────────────
# SIDEBAR
# ─────────────────────────────────────────────────────────────
with st.sidebar:
    st.markdown("## 📦 Corporacion Favorita")
    st.markdown("---")

    st.markdown('<div class="section-header">Series Info</div>', unsafe_allow_html=True)
    st.markdown(f"**Store ID:** {STORE_ID}")
    st.markdown(f"**Item ID:** {ITEM_ID}")
    st.markdown("**Model:** XGBoost (HyperOpt tuned)")
    st.markdown("**Test MAE:** 132.67 units")
    st.markdown("**Test RMSE:** 156.32 units")

    st.markdown("---")
    st.markdown('<div class="section-header">Forecast Settings</div>', unsafe_allow_html=True)

    ts = get_timeseries()
    last_date = ts.index[-1].date()
    min_forecast = last_date + timedelta(days=1)
    max_forecast = last_date + timedelta(days=90)

    forecast_start = st.date_input(
        "Forecast Start Date",
        value=min_forecast,
        min_value=min_forecast,
        max_value=max_forecast,
        help="First day to forecast. Must be after the last historical date."
    )

    n_days = st.slider(
        "Forecast Horizon (days)",
        min_value=1,
        max_value=30,
        value=14,
        step=1,
        help="Number of days to forecast autoregressively."
    )

    st.markdown("---")
    st.markdown('<div class="section-header">Chart Settings</div>', unsafe_allow_html=True)
    history_window = st.slider(
        "Historical window (days)",
        min_value=30,
        max_value=180,
        value=90,
        step=30,
        help="How many days of historical data to show in the chart."
    )

    st.markdown("---")
    st.caption("Sprint 4 | Time Series Forecasting Project")
    st.caption("Corporacion Favorita — Store 44 / Item 1047679")

# ─────────────────────────────────────────────────────────────
# MAIN HEADER
# ─────────────────────────────────────────────────────────────
st.markdown("# 📦 Corporacion Favorita — Sales Forecast")
st.markdown(
    "**Champion model:** XGBoost (HyperOpt-tuned, 50 trials) | "
    "**Store:** 44 | **Item:** 1047679 | "
    f"**Last historical date:** {last_date}"
)
st.markdown("---")

# ─────────────────────────────────────────────────────────────
# LOAD MODEL & RUN FORECAST
# ─────────────────────────────────────────────────────────────
model = get_model()

forecast_dates = pd.date_range(
    start=pd.Timestamp(forecast_start),
    periods=n_days,
    freq="D"
)

history = ts.copy()
predictions = []

for fdate in forecast_dates:
    row = make_forecast_row(fdate, history)
    pred = float(model.predict(row)[0])
    pred = max(0.0, pred)
    predictions.append(pred)
    new_entry = pd.Series([pred], index=[fdate])
    history = pd.concat([history, new_entry])

forecast_df = pd.DataFrame({
    "date":           forecast_dates,
    "forecast_sales": [round(p, 2) for p in predictions],
})
forecast_df["date"] = forecast_df["date"].dt.strftime("%Y-%m-%d")

# ─────────────────────────────────────────────────────────────
# METRIC CARDS
# ─────────────────────────────────────────────────────────────
col1, col2, col3, col4 = st.columns(4)

with col1:
    st.markdown(
        f'<div class="metric-card">'
        f'<div class="metric-label">Forecast Horizon</div>'
        f'<div class="metric-value">{n_days} days</div>'
        f'<div class="metric-sub">Starting {forecast_start}</div>'
        f'</div>', unsafe_allow_html=True
    )
with col2:
    total_pred = sum(predictions)
    st.markdown(
        f'<div class="metric-card">'
        f'<div class="metric-label">Total Forecasted Sales</div>'
        f'<div class="metric-value">{total_pred:,.0f}</div>'
        f'<div class="metric-sub">units over {n_days} days</div>'
        f'</div>', unsafe_allow_html=True
    )
with col3:
    avg_pred = np.mean(predictions)
    st.markdown(
        f'<div class="metric-card">'
        f'<div class="metric-label">Avg Daily Forecast</div>'
        f'<div class="metric-value">{avg_pred:,.1f}</div>'
        f'<div class="metric-sub">units / day</div>'
        f'</div>', unsafe_allow_html=True
    )
with col4:
    peak_idx = int(np.argmax(predictions))
    peak_date = forecast_dates[peak_idx].strftime("%b %d")
    st.markdown(
        f'<div class="metric-card">'
        f'<div class="metric-label">Peak Day</div>'
        f'<div class="metric-value">{predictions[peak_idx]:,.0f}</div>'
        f'<div class="metric-sub">{peak_date} (day {peak_idx+1})</div>'
        f'</div>', unsafe_allow_html=True
    )

st.markdown("
", unsafe_allow_html=True)

# ─────────────────────────────────────────────────────────────
# MAIN CHART — historical + forecast
# ─────────────────────────────────────────────────────────────
st.markdown('<div class="section-header">Sales History + Forecast</div>', unsafe_allow_html=True)

hist_window = ts.iloc[-history_window:]

fig = go.Figure()

# Historical actuals
fig.add_trace(go.Scatter(
    x=hist_window.index,
    y=hist_window.values,
    mode="lines",
    name="Historical Sales",
    line=dict(color="#4c9be8", width=1.5),
    opacity=0.85,
))

# Forecast
fig.add_trace(go.Scatter(
    x=forecast_dates,
    y=predictions,
    mode="lines+markers",
    name="Forecast (XGBoost)",
    line=dict(color="#f97316", width=2.5, dash="dash"),
    marker=dict(size=6, color="#f97316"),
))

# Confidence band (simple +/- 1 MAE)
mae = 132.67
upper = [p + mae for p in predictions]
lower = [max(0, p - mae) for p in predictions]

fig.add_trace(go.Scatter(
    x=list(forecast_dates) + list(reversed(forecast_dates)),
    y=upper + list(reversed(lower)),
    fill="toself",
    fillcolor="rgba(249,115,22,0.12)",
    line=dict(color="rgba(255,255,255,0)"),
    name="Confidence Band (+/- MAE)",
    showlegend=True,
))

# Vertical line at forecast start
fig.add_vline(
    x=str(forecast_dates[0].date()),
    line_width=1.5,
    line_dash="dot",
    line_color="#a0aec0",
    annotation_text="Forecast Start",
    annotation_position="top right",
    annotation_font_color="#a0aec0",
)

fig.update_layout(
    template="plotly_dark",
    paper_bgcolor="#0e1117",
    plot_bgcolor="#0e1117",
    height=420,
    margin=dict(l=10, r=10, t=30, b=10),
    legend=dict(
        orientation="h",
        yanchor="bottom", y=1.02,
        xanchor="right", x=1,
        bgcolor="rgba(0,0,0,0)",
    ),
    xaxis=dict(gridcolor="#1e2130", showgrid=True),
    yaxis=dict(gridcolor="#1e2130", showgrid=True, title="Unit Sales"),
    hovermode="x unified",
)

st.plotly_chart(fig, use_container_width=True)

# ─────────────────────────────────────────────────────────────
# FORECAST TABLE + DOWNLOAD
# ─────────────────────────────────────────────────────────────
col_left, col_right = st.columns([2, 1])

with col_left:
    st.markdown('<div class="section-header">Forecast Table</div>', unsafe_allow_html=True)
    display_df = forecast_df.copy()
    display_df.index = range(1, len(display_df) + 1)
    display_df.columns = ["Date", "Forecasted Sales (units)"]
    st.dataframe(display_df, use_container_width=True, height=min(400, 35 * n_days + 38))

with col_right:
    st.markdown('<div class="section-header">Download</div>', unsafe_allow_html=True)
    st.markdown("
", unsafe_allow_html=True)
    csv_bytes = forecast_df.to_csv(index=False).encode("utf-8")
    st.download_button(
        label="Download Forecast as CSV",
        data=csv_bytes,
        file_name=f"forecast_store{STORE_ID}_item{ITEM_ID}_{forecast_start}.csv",
        mime="text/csv",
    )
    st.markdown("
", unsafe_allow_html=True)
    st.markdown("**Model Info**")
    st.markdown(f"- Algorithm: XGBoost")
    st.markdown(f"- Tuning: HyperOpt (50 trials)")
    st.markdown(f"- Test MAE: **132.67 units**")
    st.markdown(f"- Test RMSE: **156.32 units**")
    st.markdown(f"- Features: {len(FEATURE_COLS)} engineered features")
    st.markdown(f"- Forecast type: Autoregressive")

# ─────────────────────────────────────────────────────────────
# FOOTER
# ─────────────────────────────────────────────────────────────
st.markdown("---")
st.markdown(
    "<div style='text-align:center; color:#4a5568; font-size:0.8rem;'>"
    "Corporacion Favorita Sales Forecast | Sprint 4 | "
    "XGBoost Champion Model | Store 44 / Item 1047679"
    "</div>",
    unsafe_allow_html=True
)
"""

with open(main_path, "w", encoding="utf-8") as f:
    f.write(code)

print(f"Created: app/main.py")
print(f"  Path: {main_path}")
print(f"  Size: {os.path.getsize(main_path):,} bytes")
print(f"  Lines: {code.count(chr(10))}")


Created: app/main.py
  Path: C:\Users\armin\PycharmProjects\corporacion_favorita\app\main.py
  Size: 13,683 bytes
  Lines: 337


## 7. Requirements & README


In [11]:
import os

PROJECT_ROOT = os.path.join(os.path.dirname(os.getcwd()), "corporacion_favorita")

# ── requirements.txt ──────────────────────────────────────────
lines_req = [
    "streamlit>=1.32.0\n",
    "pandas>=1.5.0\n",
    "numpy>=1.23.0\n",
    "xgboost>=2.1.0\n",
    "plotly>=5.18.0\n",
    "scikit-learn>=1.2.0\n",
    "joblib>=1.2.0\n",
    "mlflow>=2.10.0\n",
    "torch>=2.0.0\n",
    "statsmodels>=0.14.0\n",
    "hyperopt>=0.2.7\n",
]
with open(os.path.join(PROJECT_ROOT, "requirements.txt"), "w", encoding="utf-8") as f:
    f.writelines(lines_req)
print("Created: requirements.txt")

# ── README.md ─────────────────────────────────────────────────
lines_readme = [
    "# Corporacion Favorita - Sales Forecast\n\n",
    "**Store 44 / Item 1047679** | Sprint 4 - Streamlit Deployment\n\n",
    "---\n\n",
    "## Project Structure\n\n",
    "    corporacion_favorita/\n",
    "    app/config.py          - Central configuration\n",
    "    app/main.py            - Streamlit application\n",
    "    data/data_utils.py     - Data loading and feature engineering\n",
    "    model/model_utils.py   - Model loading and prediction\n",
    "    models/                - Saved model artifacts\n",
    "    mlflow_results/        - MLflow tracking store\n\n",
    "---\n\n",
    "## Model Comparison\n\n",
    "| Model | MAE | RMSE |\n",
    "|---|---|---|\n",
    "| ARIMA(1,0,1) | 181.18 | 211.01 |\n",
    "| SARIMA(1,0,1)(1,1,1,7) | 196.75 | 224.05 |\n",
    "| XGBoost (default) | 135.85 | 163.68 |\n",
    "| LSTM Baseline | 142.21 | 166.05 |\n",
    "| **XGBoost (HyperOpt)** | **132.67** | **156.32** |\n",
    "| LSTM (HyperOpt) | 155.29 | 188.34 |\n\n",
    "---\n\n",
    "## Why XGBoost Was Chosen\n\n",
    "1. Best test performance: MAE=132.67, RMSE=156.32\n",
    "2. Walk-forward backtesting: Mean MAE=105.98 +/- 9.49 (5 folds)\n",
    "3. Fast inference - ideal for real-time web app\n",
    "4. Interpretable feature importance\n",
    "5. HyperOpt tuned: n_estimators=300, max_depth=3, lr=0.01\n\n",
    "---\n\n",
    "## How to Run\n\n",
    "    pip install -r requirements.txt\n",
    "    streamlit run app/main.py\n\n",
    "App opens at http://localhost:8501\n\n",
    "---\n\n",
    "## MLflow UI\n\n",
    "    mlflow ui --backend-store-uri mlflow_results/\n\n",
    "Open http://127.0.0.1:5000\n",
    "Champion Run ID: e98efe56e2984d0aa6f0907458eca43f\n",
]
with open(os.path.join(PROJECT_ROOT, "README.md" ), "w", encoding="utf-8") as f:
    f.writelines(lines_readme)
print("Created: README.md")

# ── Verify all files ──────────────────────────────────────────
print("\n=== PROJECT FILES ===")
for root, dirs, files in os.walk(PROJECT_ROOT):
    dirs[:] = [d for d in dirs if d not in ['__pycache__', '.git', 'mlflow_results']]
    level = root.replace(PROJECT_ROOT, '').count(os.sep)
    indent = '  ' * level
    if level == 0:
        print(f"{os.path.basename(root)}/")
    else:
        print(f"{indent}{os.path.basename(root)}/")
    for file in files:
        size = os.path.getsize(os.path.join(root, file))
        print(f"{'  '*(level+1)}{file}  ({size:,} bytes)")


Created: requirements.txt
Created: README.md

=== PROJECT FILES ===
corporacion_favorita/
  README.md  (1,412 bytes)
  requirements.txt  (185 bytes)
  app/
    config.py  (1,151 bytes)
    main.py  (13,683 bytes)
    __init__.py  (61 bytes)
  data/
    data_utils.py  (2,252 bytes)
    __init__.py  (62 bytes)
  docs/
  model/
    model_utils.py  (1,513 bytes)
    __init__.py  (63 bytes)
  models/
    champion_features.json  (173 bytes)
    champion_xgboost.json  (111,202 bytes)
    scaler.pkl  (560 bytes)
    ts_store44_item1047679.csv  (30,402 bytes)


## 8. Install Streamlit
Install Streamlit and Plotly for the web application.


In [12]:
import subprocess
result = subprocess.run(
    ["pip", "install", "streamlit", "plotly", "--user", "-q"],
    capture_output=True, text=True
)
print(result.stdout[-500:] if result.stdout else "")
print(result.stderr[-300:] if result.stderr else "")
print("Done!")



  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

Done!


## 9. Launch Streamlit App (background)



In [18]:
import subprocess, os, time

PROJECT_ROOT = os.path.join(os.path.dirname(os.getcwd()), "corporacion_favorita")
streamlit_exe = os.path.expanduser(
    r"~\AppData\Roaming\Python\Python39\Scripts\streamlit.exe"
)

# Pokreni u pozadini (ne blokira Jupyter)
proc = subprocess.Popen(
    [streamlit_exe, "run", "app/main.py", "--server.headless=true"],
    cwd=PROJECT_ROOT,
    stdin=subprocess.DEVNULL,
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True
)

print(f"Streamlit PID: {proc.pid}")
print("Cekam 5 sekundi da se server pokrene...")
time.sleep(5)

# Provjeri da li jos radi
if proc.poll() is None:
    print("Streamlit radi!")
    print("Otvori browser na: http://localhost:8501" )
else:
    out, err = proc.communicate()
    print("GRESKA - Streamlit se ugasio!")
    print("STDOUT:", out[-1000:])
    print("STDERR:", err[-1000:])


Streamlit PID: 28160
Cekam 5 sekundi da se server pokrene...
Streamlit radi!
Otvori browser na: http://localhost:8501


## 10. Fix app/main.py — rewrite clean version


In [19]:
import os

PROJECT_ROOT = os.path.join(os.path.dirname(os.getcwd()), "corporacion_favorita")
main_path = os.path.join(PROJECT_ROOT, "app", "main.py")

lines = []
lines.append('import sys\n')
lines.append('import os\n')
lines.append('PROJECT_ROOT = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))\n')
lines.append('sys.path.insert(0, PROJECT_ROOT)\n')
lines.append('\n')
lines.append('import streamlit as st\n')
lines.append('import pandas as pd\n')
lines.append('import numpy as np\n')
lines.append('import plotly.graph_objects as go\n')
lines.append('from datetime import date, timedelta\n')
lines.append('\n')
lines.append('from app.config import STORE_ID, ITEM_ID, FEATURE_COLS, MODEL_PATH\n')
lines.append('from data.data_utils import load_timeseries, make_forecast_row\n')
lines.append('from model.model_utils import load_champion_model\n')
lines.append('\n')
lines.append('st.set_page_config(\n')
lines.append('    page_title="Corporacion Favorita | Sales Forecast",\n')
lines.append('    page_icon="📦",\n')
lines.append('    layout="wide",\n')
lines.append('    initial_sidebar_state="expanded",\n')
lines.append(')\n')
lines.append('\n')
lines.append('CSS = (\n')
lines.append('    "<style>"\n')
lines.append('    ".metric-card{background:linear-gradient(135deg,#1e2130,#2a2f45);"\n')
lines.append('    "border-radius:12px;padding:1.2rem 1.5rem;"\n')
lines.append('    "border-left:4px solid #4c9be8;margin-bottom:0.5rem;}"\n')
lines.append('    ".metric-label{color:#8b9dc3;font-size:0.78rem;text-transform:uppercase;}"\n')
lines.append('    ".metric-value{color:#e8eaf6;font-size:1.8rem;font-weight:700;}"\n')
lines.append('    ".metric-sub{color:#4c9be8;font-size:0.75rem;margin-top:0.2rem;}"\n')
lines.append('    ".section-header{color:#4c9be8;font-size:1.05rem;font-weight:600;"\n')
lines.append('    "text-transform:uppercase;letter-spacing:0.06em;"\n')
lines.append('    "border-bottom:1px solid #2a2f45;padding-bottom:0.3rem;margin-bottom:0.5rem;}"\n')
lines.append('    "</style>"\n')
lines.append(')\n')
lines.append('st.markdown(CSS, unsafe_allow_html=True)\n')
lines.append('\n')
lines.append('@st.cache_resource(show_spinner="Loading champion model...")\n')
lines.append('def get_model():\n')
lines.append('    return load_champion_model()\n')
lines.append('\n')
lines.append('@st.cache_data(show_spinner="Loading historical data...")\n')
lines.append('def get_timeseries():\n')
lines.append('    return load_timeseries()\n')
lines.append('\n')
lines.append('ts = get_timeseries()\n')
lines.append('last_date = ts.index[-1].date()\n')
lines.append('min_forecast = last_date + timedelta(days=1)\n')
lines.append('max_forecast = last_date + timedelta(days=90)\n')
lines.append('\n')
lines.append('with st.sidebar:\n')
lines.append('    st.markdown("## Corporacion Favorita")\n')
lines.append('    st.markdown("---")\n')
lines.append('    st.markdown("**Store ID:** " + str(STORE_ID))\n')
lines.append('    st.markdown("**Item ID:** " + str(ITEM_ID))\n')
lines.append('    st.markdown("**Model:** XGBoost (HyperOpt tuned)")\n')
lines.append('    st.markdown("**Test MAE:** 132.67 units")\n')
lines.append('    st.markdown("**Test RMSE:** 156.32 units")\n')
lines.append('    st.markdown("---")\n')
lines.append('    forecast_start = st.date_input(\n')
lines.append('        "Forecast Start Date",\n')
lines.append('        value=min_forecast,\n')
lines.append('        min_value=min_forecast,\n')
lines.append('        max_value=max_forecast,\n')
lines.append('    )\n')
lines.append('    n_days = st.slider("Forecast Horizon (days)", 1, 30, 14)\n')
lines.append('    history_window = st.slider("Historical window (days)", 30, 180, 90, 30)\n')
lines.append('    st.caption("Sprint 4 | Store 44 / Item 1047679")\n')
lines.append('\n')
lines.append('st.markdown("# Corporacion Favorita — Sales Forecast")\n')
lines.append('st.markdown(\n')
lines.append('    "**Champion model:** XGBoost (HyperOpt, 50 trials) | "\n')
lines.append('    "**Store:** 44 | **Item:** 1047679 | "\n')
lines.append('    "**Last historical date:** " + str(last_date)\n')
lines.append(')\n')
lines.append('st.markdown("---")\n')
lines.append('\n')
lines.append('model = get_model()\n')
lines.append('\n')
lines.append('forecast_dates = pd.date_range(start=pd.Timestamp(forecast_start), periods=n_days, freq="D")\n')
lines.append('history = ts.copy()\n')
lines.append('predictions = []\n')
lines.append('for fdate in forecast_dates:\n')
lines.append('    row = make_forecast_row(fdate, history)\n')
lines.append('    pred = max(0.0, float(model.predict(row)[0]))\n')
lines.append('    predictions.append(pred)\n')
lines.append('    history = pd.concat([history, pd.Series([pred], index=[fdate])])\n')
lines.append('\n')
lines.append('forecast_df = pd.DataFrame({"date": forecast_dates.strftime("%Y-%m-%d"), "forecast_sales": [round(p,2) for p in predictions]})\n')
lines.append('\n')
lines.append('col1, col2, col3, col4 = st.columns(4)\n')
lines.append('with col1:\n')
lines.append('    st.metric("Forecast Horizon", str(n_days) + " days")\n')
lines.append('with col2:\n')
lines.append('    st.metric("Total Forecasted Sales", f"{sum(predictions):,.0f} units")\n')
lines.append('with col3:\n')
lines.append('    st.metric("Avg Daily Forecast", f"{np.mean(predictions):,.1f} units")\n')
lines.append('with col4:\n')
lines.append('    peak_idx = int(np.argmax(predictions))\n')
lines.append('    st.metric("Peak Day", f"{predictions[peak_idx]:,.0f} units", forecast_dates[peak_idx].strftime("%b %d"))\n')
lines.append('\n')
lines.append('st.markdown("---")\n')
lines.append('st.markdown("### Sales History + Forecast")\n')
lines.append('\n')
lines.append('hist_window = ts.iloc[-history_window:]\n')
lines.append('mae = 132.67\n')
lines.append('upper = [p + mae for p in predictions]\n')
lines.append('lower = [max(0, p - mae) for p in predictions]\n')
lines.append('\n')
lines.append('fig = go.Figure()\n')
lines.append('fig.add_trace(go.Scatter(x=hist_window.index, y=hist_window.values, mode="lines", name="Historical Sales", line=dict(color="#4c9be8", width=1.5)))\n')
lines.append('fig.add_trace(go.Scatter(x=forecast_dates, y=predictions, mode="lines+markers", name="Forecast (XGBoost)", line=dict(color="#f97316", width=2.5, dash="dash"), marker=dict(size=6)))\n')
lines.append('fig.add_trace(go.Scatter(\n')
lines.append('    x=list(forecast_dates) + list(reversed(forecast_dates)),\n')
lines.append('    y=upper + list(reversed(lower)),\n')
lines.append('    fill="toself", fillcolor="rgba(249,115,22,0.12)",\n')
lines.append('    line=dict(color="rgba(255,255,255,0)"),\n')
lines.append('    name="Confidence Band (+/- MAE)"\n')
lines.append('))\n')
lines.append('fig.add_vline(x=str(forecast_dates[0].date()), line_width=1.5, line_dash="dot", line_color="#a0aec0", annotation_text="Forecast Start", annotation_font_color="#a0aec0")\n')
lines.append('fig.update_layout(template="plotly_dark", paper_bgcolor="#0e1117", plot_bgcolor="#0e1117", height=420,\n')
lines.append('    margin=dict(l=10,r=10,t=30,b=10),\n')
lines.append('    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1, bgcolor="rgba(0,0,0,0)"),\n')
lines.append('    xaxis=dict(gridcolor="#1e2130"), yaxis=dict(gridcolor="#1e2130", title="Unit Sales"),\n')
lines.append('    hovermode="x unified")\n')
lines.append('st.plotly_chart(fig, use_container_width=True)\n')
lines.append('\n')
lines.append('col_left, col_right = st.columns([2, 1])\n')
lines.append('with col_left:\n')
lines.append('    st.markdown("### Forecast Table")\n')
lines.append('    display_df = forecast_df.copy()\n')
lines.append('    display_df.index = range(1, len(display_df)+1)\n')
lines.append('    display_df.columns = ["Date", "Forecasted Sales (units)"]\n')
lines.append('    st.dataframe(display_df, use_container_width=True)\n')
lines.append('with col_right:\n')
lines.append('    st.markdown("### Download")\n')
lines.append('    csv_bytes = forecast_df.to_csv(index=False).encode("utf-8")\n')
lines.append('    st.download_button("Download Forecast as CSV", csv_bytes,\n')
lines.append('        file_name="forecast_store44_item1047679.csv", mime="text/csv")\n')
lines.append('    st.markdown("**Model Info**")\n')
lines.append('    st.markdown("- Algorithm: XGBoost")\n')
lines.append('    st.markdown("- Tuning: HyperOpt (50 trials)")\n')
lines.append('    st.markdown("- Test MAE: **132.67 units**")\n')
lines.append('    st.markdown("- Test RMSE: **156.32 units**")\n')
lines.append('    st.markdown("- Features: " + str(len(FEATURE_COLS)) + " engineered features")\n')
lines.append('\n')
lines.append('st.markdown("---")\n')
lines.append('st.caption("Corporacion Favorita Sales Forecast | Sprint 4 | XGBoost Champion | Store 44 / Item 1047679")\n')

with open(main_path, "w", encoding="utf-8") as f:
    f.writelines(lines)

print(f"Rewrote: app/main.py")
print(f"  Lines: {len(lines)}")
print(f"  Size:  {os.path.getsize(main_path):,} bytes")


Rewrote: app/main.py
  Lines: 148
  Size:  6,064 bytes


## 11. Fix model/model_utils.py — load directly from local JSON


In [20]:
import os

PROJECT_ROOT = os.path.join(os.path.dirname(os.getcwd()), "corporacion_favorita")
model_utils_path = os.path.join(PROJECT_ROOT, "model", "model_utils.py")

lines = []
lines.append('import os\n')
lines.append('import sys\n')
lines.append('import pandas as pd\n')
lines.append('import xgboost as xgb\n')
lines.append('\n')
lines.append('sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))\n')
lines.append('from app.config import MODEL_PATH\n')
lines.append('\n')
lines.append('def load_champion_model():\n')
lines.append('    model = xgb.XGBRegressor()\n')
lines.append('    model.load_model(MODEL_PATH)\n')
lines.append('    return model\n')
lines.append('\n')
lines.append('def predict_one(model, feature_row: pd.DataFrame) -> float:\n')
lines.append('    pred = model.predict(feature_row)\n')
lines.append('    return float(pred[0])\n')

with open(model_utils_path, "w", encoding="utf-8") as f:
    f.writelines(lines)

print(f"Rewrote: model/model_utils.py")
print(f"  Lines: {len(lines)}")

# Provjeri da li model fajl postoji
from app.config import MODEL_PATH
print(f"  MODEL_PATH: {MODEL_PATH}")
print(f"  Model file exists: {os.path.exists(MODEL_PATH)}")


Rewrote: model/model_utils.py
  Lines: 16


ModuleNotFoundError: No module named 'app'

## 12. Restart Streamlit server


In [21]:
import subprocess, os, time

# Ubij stari server
subprocess.run(["taskkill", "/F", "/IM", "streamlit.exe"], capture_output=True)
time.sleep(2)

PROJECT_ROOT = os.path.join(os.path.dirname(os.getcwd()), "corporacion_favorita")
streamlit_exe = os.path.expanduser(
    r"~\AppData\Roaming\Python\Python39\Scripts\streamlit.exe"
)

proc = subprocess.Popen(
    [streamlit_exe, "run", "app/main.py", "--server.headless=true"],
    cwd=PROJECT_ROOT,
    stdin=subprocess.DEVNULL,
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
)

print(f"Streamlit PID: {proc.pid}")
time.sleep(6)

if proc.poll() is None:
    print("Streamlit radi!")
    print("Otvori: http://localhost:8501" )
else:
    out, err = proc.communicate()
    print("GRESKA:")
    print(out.decode("utf-8", errors="ignore")[-1000:])
    print(err.decode("utf-8", errors="ignore")[-1000:])


Streamlit PID: 48288
Streamlit radi!
Otvori: http://localhost:8501


## 13. Check Streamlit logs


In [22]:
import subprocess, os, time

# Ubij stari server
subprocess.run(["taskkill", "/F", "/IM", "streamlit.exe"], capture_output=True)
time.sleep(2)

PROJECT_ROOT = os.path.join(os.path.dirname(os.getcwd()), "corporacion_favorita")
streamlit_exe = os.path.expanduser(
    r"~\AppData\Roaming\Python\Python39\Scripts\streamlit.exe"
)

proc = subprocess.Popen(
    [streamlit_exe, "run", "app/main.py", "--server.headless=true"],
    cwd=PROJECT_ROOT,
    stdin=subprocess.DEVNULL,
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
)

print(f"PID: {proc.pid}")
time.sleep(15)

# Citaj output
import threading
out_lines = []
err_lines = []

def read_out():
    for line in proc.stdout:
        out_lines.append(line.decode("utf-8", errors="ignore"))

def read_err():
    for line in proc.stderr:
        err_lines.append(line.decode("utf-8", errors="ignore"))

t1 = threading.Thread(target=read_out, daemon=True)
t2 = threading.Thread(target=read_err, daemon=True)
t1.start(); t2.start()
time.sleep(5)

print("=== STDOUT ===")
print("".join(out_lines[-30:]))
print("=== STDERR ===")
print("".join(err_lines[-30:]))


PID: 37896
=== STDOUT ===



  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://192.168.2.38:8501
  External URL: http://84.136.17.247:8501


=== STDERR ===



## 15. Fix app/main.py — replace add_vline


In [24]:
import os

PROJECT_ROOT = os.path.join(os.path.dirname(os.getcwd()), "corporacion_favorita")
main_path = os.path.join(PROJECT_ROOT, "app", "main.py")

with open(main_path, "r", encoding="utf-8") as f:
    content = f.read()

# Zamijeni add_vline sa Scatter vertical line
old_line = 'fig.add_vline(x=str(forecast_dates[0].date()), line_width=1.5, line_dash="dot", line_color="#a0aec0", annotation_text="Forecast Start", annotation_font_color="#a0aec0")'

new_line = (
    'fig.add_trace(go.Scatter(\n'
    '    x=[forecast_dates[0], forecast_dates[0]],\n'
    '    y=[0, hist_window.max() * 1.2],\n'
    '    mode="lines",\n'
    '    name="Forecast Start",\n'
    '    line=dict(color="#a0aec0", width=1.5, dash="dot"),\n'
    '    showlegend=False,\n'
    '))'
)

if old_line in content:
    content = content.replace(old_line, new_line)
    with open(main_path, "w", encoding="utf-8") as f:
        f.write(content)
    print("Fixed: add_vline replaced with Scatter trace")
else:
    print("Line not found — printing lines around 120:")
    lines = content.split("\n")
    for i, l in enumerate(lines[115:125], start=116):
        print(f"{i}: {l}")


Line not found — printing lines around 120:
116:     fill="toself", fillcolor="rgba(249,115,22,0.12)",
117:     line=dict(color="rgba(255,255,255,0)"),
118:     name="Confidence Band (+/- MAE)"
119: ))
120: fig.add_trace(go.Scatter(
121:     x=[forecast_dates[0], forecast_dates[0]],
122:     y=[0, hist_window.max() * 1.2],
123:     mode="lines",
124:     name="Forecast Start",
125:     line=dict(color="#a0aec0", width=1.5, dash="dot"),


## 16. Restart Streamlit after fix


In [25]:
import subprocess, os, time

subprocess.run(["taskkill", "/F", "/IM", "streamlit.exe"], capture_output=True)
time.sleep(2)

PROJECT_ROOT = os.path.join(os.path.dirname(os.getcwd()), "corporacion_favorita")
streamlit_exe = os.path.expanduser(r"~\AppData\Roaming\Python\Python39\Scripts\streamlit.exe")

proc = subprocess.Popen(
    [streamlit_exe, "run", "app/main.py", "--server.headless=true"],
    cwd=PROJECT_ROOT,
    stdin=subprocess.DEVNULL,
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
)

print(f"PID: {proc.pid}")
time.sleep(5)
if proc.poll() is None:
    print("Streamlit radi! Otvori: http://localhost:8501" )
else:
    out, err = proc.communicate()
    print("GRESKA:", out.decode("utf-8","ignore")[-500:])
    print(err.decode("utf-8","ignore")[-500:])


PID: 23844
Streamlit radi! Otvori: http://localhost:8501


## 17. Sprint 4 — Summary & Screenshots


In [26]:
print("=" * 60)
print("SPRINT 4 — DEPLOYMENT COMPLETE")
print("=" * 60)
print()
print("Files created:")
import os
PROJECT_ROOT = os.path.join(os.path.dirname(os.getcwd()), "corporacion_favorita")
for root, dirs, files in os.walk(PROJECT_ROOT):
    dirs[:] = [d for d in dirs if d not in ['__pycache__', '.git', 'mlflow_results']]
    level = root.replace(PROJECT_ROOT, '').count(os.sep)
    indent = '  ' * level
    if level == 0:
        print(f"{os.path.basename(root)}/")
    else:
        print(f"{indent}{os.path.basename(root)}/")
    for file in files:
        size = os.path.getsize(os.path.join(root, file))
        print(f"{'  '*(level+1)}{file}  ({size:,} bytes)")

print()
print("Streamlit app: http://localhost:8501" )
print()
print("Champion model: XGBoost (HyperOpt-tuned)")
print("  MAE  = 132.67 units")
print("  RMSE = 156.32 units")
print("  Walk-forward MAE = 105.98 +/- 9.49 (5 folds)")
print()
print("App features:")
print("  - Sidebar: store info, date picker, forecast horizon slider")
print("  - Metric cards: total sales, avg daily, peak day")
print("  - Interactive Plotly chart: history + forecast + confidence band")
print("  - Forecast table with day-by-day predictions")
print("  - CSV download button")
print("  - Dark theme")
print()
print("Sprint 4 DONE!")


SPRINT 4 — DEPLOYMENT COMPLETE

Files created:
corporacion_favorita/
  README.md  (1,412 bytes)
  requirements.txt  (185 bytes)
  app/
    config.py  (1,151 bytes)
    main.py  (6,135 bytes)
    __init__.py  (61 bytes)
  data/
    data_utils.py  (2,252 bytes)
    __init__.py  (62 bytes)
  docs/
  model/
    model_utils.py  (428 bytes)
    __init__.py  (63 bytes)
  models/
    champion_features.json  (173 bytes)
    champion_xgboost.json  (111,202 bytes)
    scaler.pkl  (560 bytes)
    ts_store44_item1047679.csv  (30,402 bytes)

Streamlit app: http://localhost:8501

Champion model: XGBoost (HyperOpt-tuned)
  MAE  = 132.67 units
  RMSE = 156.32 units
  Walk-forward MAE = 105.98 +/- 9.49 (5 folds)

App features:
  - Sidebar: store info, date picker, forecast horizon slider
  - Metric cards: total sales, avg daily, peak day
  - Interactive Plotly chart: history + forecast + confidence band
  - Forecast table with day-by-day predictions
  - CSV download button
  - Dark theme

Sprint 4 DONE!

## Sprint 4 — Complete ✓

### Deliverables

| File | Description |
|---|---|
| `app/config.py` | Central configuration — paths, constants, feature list |
| `app/main.py` | Streamlit web application (337 lines) |
| `data/data_utils.py` | Data loading and autoregressive feature engineering |
| `model/model_utils.py` | Champion model loader (XGBoost local JSON) |
| `models/champion_xgboost.json` | Saved XGBoost champion model (108 KB) |
| `requirements.txt` | Python dependencies |
| `README.md` | Project documentation with model rationale |

### App Features
- **Sidebar:** Store/Item info, forecast date picker, horizon slider (1–30 days), historical window slider
- **Metric cards:** Forecast horizon, total forecasted sales, average daily forecast, peak day
- **Interactive chart:** Historical sales (blue) + autoregressive forecast (orange dashed) + confidence band (±MAE)
- **Forecast table:** Day-by-day predictions with numbered rows
- **CSV download:** Export forecast results
- **Dark theme:** Plotly dark template

### Champion Model Summary
| Metric | Value |
|---|---|
| Algorithm | XGBoost (HyperOpt-tuned, 50 trials) |
| Test MAE | 132.67 units |
| Test RMSE | 156.32 units |
| Walk-forward MAE | 105.98 ± 9.49 (5 folds) |
| Features | 13 engineered (lags, rolling stats, calendar) |

### How to Run
```bash
streamlit run app/main.py


## 18. GitHub — Create .gitignore and push to repository


In [28]:
import os

PROJECT_ROOT = os.path.join(os.path.dirname(os.getcwd()), "corporacion_favorita")

lines = [
    "# MLflow store (large local files)\n",
    "mlflow_results/\n",
    "\n",
    "# Python caches\n",
    "__pycache__/\n",
    "*.pyc\n",
    "*.pyo\n",
    ".ipynb_checkpoints/\n",
    "\n",
    "# OS files\n",
    ".DS_Store\n",
    "Thumbs.db\n",
    ".idea/\n",
    ".vscode/\n",
    "\n",
    "# Streamlit secrets\n",
    ".streamlit/secrets.toml\n",
    ".env\n",
]

with open(os.path.join(PROJECT_ROOT, ".gitignore"), "w", encoding="utf-8") as f:
    f.writelines(lines)

print("Created: .gitignore")


Created: .gitignore


In [29]:
import subprocess, os

PROJECT_ROOT = os.path.join(os.path.dirname(os.getcwd()), "corporacion_favorita")

def run(cmd):
    result = subprocess.run(cmd, cwd=PROJECT_ROOT, capture_output=True, text=True, shell=True)
    out = result.stdout.strip()
    err = result.stderr.strip()
    if out: print(out)
    if err: print(err)
    return result.returncode

print("=== Git setup ===")
run('git init')
run('git branch -M main')
run('git config user.email "kajevic44@gmail.com"')
run('git config user.name "kajevic44-sudo"')
run('git remote remove origin')
run('git remote add origin https://github.com/kajevic44-sudo/corporacion_favorita.git' )
print()
print("=== Staging files ===")
run('git add .')
run('git status')
print()
print("=== Commit ===")
run('git commit -m "Sprint 4: Streamlit forecasting app with XGBoost champion model"')
print()
print("=== Push ===")
rc = run('git push -u origin main')
if rc == 0:
    print("\nPush successful!")
    print("https://github.com/kajevic44-sudo/corporacion_favorita" )
else:
    print("\nPush failed — vjerovatno treba GitHub token (PAT). Javi mi output!")


=== Git setup ===
Der Befehl "git" ist entweder falsch geschrieben oder
konnte nicht gefunden werden.
Der Befehl "git" ist entweder falsch geschrieben oder
konnte nicht gefunden werden.
Der Befehl "git" ist entweder falsch geschrieben oder
konnte nicht gefunden werden.
Der Befehl "git" ist entweder falsch geschrieben oder
konnte nicht gefunden werden.
Der Befehl "git" ist entweder falsch geschrieben oder
konnte nicht gefunden werden.
Der Befehl "git" ist entweder falsch geschrieben oder
konnte nicht gefunden werden.

=== Staging files ===
Der Befehl "git" ist entweder falsch geschrieben oder
konnte nicht gefunden werden.
Der Befehl "git" ist entweder falsch geschrieben oder
konnte nicht gefunden werden.

=== Commit ===
Der Befehl "git" ist entweder falsch geschrieben oder
konnte nicht gefunden werden.

=== Push ===
Der Befehl "git" ist entweder falsch geschrieben oder
konnte nicht gefunden werden.

Push failed — vjerovatno treba GitHub token (PAT). Javi mi output!


## 19. Install Git for Windows


In [30]:
import subprocess, os

# Provjeri da li git postoji
result = subprocess.run("where git", capture_output=True, text=True, shell=True)
print("Git lokacija:", result.stdout.strip() if result.stdout.strip() else "NIJE PRONADJEN")

# Provjeri da li winget postoji (Windows package manager)
result2 = subprocess.run("winget --version", capture_output=True, text=True, shell=True)
print("Winget:", result2.stdout.strip() if result2.stdout.strip() else "nije dostupan")

# Provjeri da li choco postoji
result3 = subprocess.run("choco --version", capture_output=True, text=True, shell=True)
print("Chocolatey:", result3.stdout.strip() if result3.stdout.strip() else "nije dostupan")


Git lokacija: NIJE PRONADJEN
Winget: v1.28.240
Chocolatey: nije dostupan


## 20. Install Git via winget


In [31]:
import subprocess, time

print("Instaliram Git... (moze potrajati 1-2 minute)")
result = subprocess.run(
    "winget install --id Git.Git -e --source winget --accept-package-agreements --accept-source-agreements",
    capture_output=True, text=True, shell=True, timeout=180
)
print(result.stdout[-1000:] if result.stdout else "")
print(result.stderr[-500:] if result.stderr else "")
print("Return code:", result.returncode)


Instaliram Git... (moze potrajati 1-2 minute)
\ 
   | 
   / 
   - 
   \ 
   | 
   / 
   - 
   \ 
   | 
   / 
   - 
   \ 
   | 
   / 
   - 
   \ 
   | 
   / 
   - 
   \ 
   | 
   / 
   - 
   \ 
   | 
   / 
   - 
   \ 
   | 
   / 
   - 
   \ 
   | 
   / 
   - 
   \ 
   | 
   / 
   - 
   \ 
   | 
   / 
   - 
   \ 
   | 
   / 
   - 
   \ 
   | 
   / 
   - 
   \ 
   | 
   / 
   - 
   \ 
   | 
   / 
   - 
   \ 
   | 
   / 
   - 
   \ 
   | 
   / 
   - 
   \ 
   | 
   / 
   - 
   \ 
   | 
   / 
   - 
   \ 
   | 
   / 
   - 
   \ 
   | 
   / 
   - 
   \ 
   | 
   / 
   - 
   \ 
   | 
   / 
   - 
   \ 
   | 
   / 
   - 
   \ 
   | 
   / 
   - 
   \ 
   | 
   / 
   - 
   \ 
   | 
   / 
   - 
   \ 
   | 
   / 
   - 
   \ 
   | 
   / 
   - 
   \ 
   | 
   / 
   - 
   \ 
   | 
   / 
   - 
   \ 
   | 
   / 
   - 
   \ 
   | 
   / 
   - 
   \ 
   | 
   / 
   - 
   \ 
   | 
   / 
   - 
   \ 
   | 
   / 
                                                                                                   

## 21. Push to GitHub


In [ ]:
import subprocess, os, sys

PROJECT_ROOT = os.path.join(os.path.dirname(os.getcwd()), "corporacion_favorita")

# Git putanja nakon instalacije
GIT = r"C:\Program Files\Git\bin\git.exe"

def run(cmd_args):
    result = subprocess.run(
        [GIT] + cmd_args,
        cwd=PROJECT_ROOT,
        capture_output=True,
        text=True
    )
    out = result.stdout.strip()
    err = result.stderr.strip()
    if out: print(out)
    if err: print(err)
    return result.returncode

print("Git version:")
run(["--version"])

print("\n=== Git setup ===")
run(["init"])
run(["branch", "-M", "main"])
run(["config", "user.email", "kajevic44@gmail.com"])
run(["config", "user.name", "kajevic44-sudo"])
run(["remote", "remove", "origin"])
run(["remote", "add", "origin", "https://github.com/kajevic44-sudo/corporacion_favorita.git"] )

print("\n=== Staging ===")
run(["add", "."])
run(["status"])

print("\n=== Commit ===")
run(["commit", "-m", "Sprint 4: Streamlit forecasting app with XGBoost champion model"])

print("\n=== Push ===")
rc = run(["push", "-u", "origin", "main"])
if rc == 0:
    print("\nPush USPJESAN!")
    print("https://github.com/kajevic44-sudo/corporacion_favorita" )
else:
    print("\nPush nije uspio — treba GitHub Personal Access Token (PAT).")
    print("Javi mi output!")


Git version:
git version 2.54.0.windows.1

=== Git setup ===
Initialized empty Git repository in C:/Users/armin/PycharmProjects/corporacion_favorita/.git/
error: No such remote: 'origin'

=== Staging ===
On branch main

No commits yet

Changes to be committed:
  (use "git rm --cached <file>..." to unstage)
	new file:   .gitignore
	new file:   README.md
	new file:   app/__init__.py
	new file:   app/config.py
	new file:   app/main.py
	new file:   data/__init__.py
	new file:   data/data_utils.py
	new file:   model/__init__.py
	new file:   model/model_utils.py
	new file:   models/champion_features.json
	new file:   models/champion_xgboost.json
	new file:   models/scaler.pkl
	new file:   models/ts_store44_item1047679.csv
	new file:   requirements.txt

=== Commit ===
[main (root-commit) 602a02a] Sprint 4: Streamlit forecasting app with XGBoost champion model
 14 files changed, 2031 insertions(+)
 create mode 100644 .gitignore
 create mode 100644 README.md
 create mode 100644 app/__init__.py


## 22. Push to GitHub with token


In [1]:
import subprocess, os

PROJECT_ROOT = os.path.join(os.path.dirname(os.getcwd()), "corporacion_favorita")
GIT = r"C:\Program Files\Git\bin\git.exe"

def run(cmd_args):
    result = subprocess.run(
        [GIT] + cmd_args,
        cwd=PROJECT_ROOT,
        capture_output=True,
        text=True
    )
    out = result.stdout.strip()
    err = result.stderr.strip()
    if out: print(out)
    if err: print(err)
    return result.returncode

TOKEN = "YOUR_GITHUB_TOKEN_HERE"
REMOTE_URL = f"https://{TOKEN}@github.com/kajevic44-sudo/corporacion_favorita.git"

run(["remote", "remove", "origin"] )
run(["remote", "add", "origin", REMOTE_URL])

print("Pushing to GitHub...")
rc = run(["push", "-u", "origin", "main"])
if rc == 0:
    print("\nPush USPJESAN!")
    print("https://github.com/kajevic44-sudo/corporacion_favorita" )
else:
    print("\nGreska!")


Pushing to GitHub...
branch 'main' set up to track 'origin/main'.
To https://github.com/kajevic44-sudo/corporacion_favorita.git
 * [new branch]      main -> main

Push USPJESAN!
https://github.com/kajevic44-sudo/corporacion_favorita


## 23. Add Sprint notebooks to GitHub
